In [3]:
# Welcome to your new notebook
# Type here in the cell editor to add code!

df = spark.read.table("s_sales_transactions")

from pyspark.sql.functions import col, sum, avg, count, round, max, min, month, year, date_format

gold_customer = df.groupBy("customer_id").agg(
        count("transaction_id").alias("total_orders"),
        round(sum("net_revenue"), 2).alias("total_revenue"),
        round(avg("net_revenue"), 2).alias("avg_order_value"),
        round(sum("discount_amount"), 2).alias("total_discount_given"),
        sum("quantity").alias("total_items_bought")
    ).orderBy(col("total_revenue").desc())

display(gold_customer)


gold_product = df.groupBy("product_id").agg(
        count("transaction_id").alias("total_orders"),
        sum("quantity").alias("total_quantity_sold"),
        round(sum("net_revenue"), 2).alias("total_revenue"),
        round(avg("unit_price"), 2).alias("avg_unit_price"),
        round(sum(col("is_returned").cast("int")), 2).alias("total_returns")
    ).orderBy(col("total_revenue").desc())

display(gold_product)


gold_monthly = df.groupBy(
        year("order_date").alias("year"),
        month("order_date").alias("month"),
        date_format("order_date", "yyyy-MM").alias("year_month")
    ).agg(
        count("transaction_id").alias("total_orders"),
        round(sum("gross_revenue"), 2).alias("total_gross_revenue"),
        round(sum("discount_amount"), 2).alias("total_discount"),
        round(sum("net_revenue"), 2).alias("total_net_revenue"),
        round(avg("net_revenue"), 2).alias("avg_order_value")
    ).orderBy("year", "month")  # chronological order

display(gold_monthly)

from pyspark.sql.functions import when, col

gold_monthly = gold_monthly.withColumn("month_name",
    when(col("month") == 1, "Jan")
    .when(col("month") == 2, "Feb")
    .when(col("month") == 3, "Mar")
    .when(col("month") == 4, "Apr")
    .when(col("month") == 5, "May")
    .when(col("month") == 6, "Jun")
    .when(col("month") == 7, "Jul")
    .when(col("month") == 8, "Aug")
    .when(col("month") == 9, "Sep")
    .when(col("month") == 10, "Oct")
    .when(col("month") == 11, "Nov")
    .when(col("month") == 12, "Dec")
)

gold_customer.write.format("delta").mode("overwrite").saveAsTable("gold_customer_summary")
gold_product.write.format("delta").mode("overwrite").saveAsTable("gold_product_summary")
gold_monthly.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_monthly_summary")

print("All gold tables saved successfully!")

StatementMeta(, ed04e012-cb66-41fa-80b8-75172977cdb8, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4728db21-dba1-4b45-8334-4737d0886b8f)

SynapseWidget(Synapse.DataFrame, 44896f9c-b6bf-42ec-92f5-6817839f035c)

SynapseWidget(Synapse.DataFrame, 38ac7751-f8bc-429d-bde3-0872dc29c44e)

All gold tables saved successfully!
